# Dubois surface inversion evaluation

In this notebook we compare results of the new implementation of the Dubois surface inversion with the C legacy version of PolSARpro.  

In [ ]:
%load_ext autoreload
%autoreload 2

import os

from polsarpro.dev.devtools import parse_psp_parameter_string
import numpy as np
from pathlib import Path
import xarray as xr
from polsarpro.io import open_netcdf_beam
from polsarpro.dev.io import read_psp_bin
from polsarpro.physical_inversion import dubois_surface_inversion
from polsarpro.dev.metrics import summarize_metrics, visualize_errors
import shutil

# optional import for progress bar
from dask.diagnostics import ProgressBar

# change to your local C-PolSARpro install dir
c_psp_dir = "/home/c_psp/Soft/bin/"
os.environ["PATH"]+=os.pathsep+f"{c_psp_dir}/data_process_sngl/"
os.environ["PATH"]+=os.pathsep+f"{c_psp_dir}/data_convert/"

# change to your data paths
# original dataset
input_alos_data = Path("/data/psp/test_files/SAN_FRANCISCO_ALOS1_slc.nc")

# S2 inputs generated with C-PSP
input_test_dir = Path("/data/psp/SAN_FRANCISCO_ALOS1/")

# output files from C
output_test_dir = Path("/data/psp/res/dubois_c/")
if not os.path.isdir(output_test_dir):
    os.mkdir(output_test_dir)

# Made-up Dubois parameters for this comparison notebook.
freq_ghz = 1.27
angle_unit = 0  # C-PolSARpro: 0 = degrees, 1 = radians
calibration_flag = 1
calibration_coeff = 1.0
thresh1_db = 0.0
thresh2_db = -10.0
fnr = 18432
fnc = 1248

## Run the C-version on some test data

In [ ]:
# Create a synthetic incidence angle file for C-PolSARpro.
# The C executable reads float32 values in row-major order and converts
# degrees to radians when -un is 0.
incidence_angle_file = output_test_dir / "dubois_incidence_angle_deg.bin"
incidence_angle_deg = np.broadcast_to(
    np.linspace(22.0, 27.0, fnc, dtype=np.float32),
    (fnr, fnc),
)
incidence_angle_deg.tofile(incidence_angle_file)

input_str = f"""id: {input_test_dir}
od: {output_test_dir}
iodf: S2
ang: {incidence_angle_file}
ofr: 0
ofc: 0
fnr: {fnr}
fnc: {fnc}
fr: {freq_ghz}
un: {angle_unit}
caf: {calibration_flag}
cac: {calibration_coeff}
th1: {thresh1_db}
th2: {thresh2_db}
errf: /tmp"""

result = parse_psp_parameter_string(input_str)
os.system(f"surface_inversion_dubois.exe {result}")

# CPSP does not create another config file in output dir
shutil.copy(input_test_dir / "config.txt", output_test_dir)

## Load ALOS data and C outputs

In [ ]:
# uncomment to test on S matrix made with SNAP
S = open_netcdf_beam(input_alos_data)

## Apply the surface inversion

In [ ]:
row_dim, col_dim = S.dims
incidence_angle = xr.DataArray(
    np.broadcast_to(
        np.deg2rad(np.linspace(22.0, 27.0, S.sizes[col_dim], dtype=np.float32)),
        (S.sizes[row_dim], S.sizes[col_dim]),
    ),
    dims=(row_dim, col_dim),
    coords={row_dim: S.coords[row_dim], col_dim: S.coords[col_dim]},
    name="incidence_angle",
)

file_out = "/data/psp/res/test_dubois_surface_inversion.nc"
# netcdf writer cannot overwrite
if os.path.isfile(file_out):
    os.remove(file_out)

with ProgressBar():
    dubois_surface_inversion(
        S,
        incidence_angle=incidence_angle,
        freq_ghz=freq_ghz,
        thresh1=thresh1_db,
        thresh2=thresh2_db,
        calibration_coeff=calibration_coeff,
    ).to_netcdf(file_out)

# Numerical evaluation

In [ ]:
import matplotlib.pyplot as plt
# open python output (comment if using compute)
out_py = xr.open_dataset("/data/psp/res/test_dubois_surface_inversion.nc")
# open C-PolSARPro outputs
out_c = {}
out_names = (
    "dubois_er",
    "dubois_mv",
    "dubois_ks",
    "dubois_mask_out",
    "dubois_mask_in",
    "dubois_mask_valid_in_out",
)
for name in out_names:
    file_name = output_test_dir / f"{name}.bin"
    out_c[name] = read_psp_bin(file_name)

df = summarize_metrics(out_py, out_c, short_titles=False, verbose=False)
df

In [ ]:
visualize_errors(out_py=out_py, out_c=out_c)